In [ ]:
# Core scientific/imaging stack.
#!pip install -q opencv-python-headless scipy matplotlib
import cv2                      # OpenCV — thresholding, contours, affine warps
import numpy as np             # array math
import matplotlib.pyplot as plt

print('OpenCV version :', cv2.__version__)
print('NumPy  version :', np.__version__)

# A tiny helper so every figure in this notebook looks consistent.
def show(images, titles=None, cols=4, cmap='gray', figsize=None):
    """Display one or many images in a grid (BGR images are auto-converted)."""
    if isinstance(images, np.ndarray) and images.ndim in (2, 3) and not isinstance(titles, (list, tuple)):
        images = [images]
    n = len(images)
    cols = min(cols, n)
    rows = int(np.ceil(n / cols))
    figsize = figsize or (3 * cols, 3 * rows)
    plt.figure(figsize=figsize)
    for i, img in enumerate(images):
        ax = plt.subplot(rows, cols, i + 1)
        disp = img
        if img.ndim == 3:                      # OpenCV is BGR, matplotlib expects RGB
            disp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(disp, cmap=cmap if img.ndim == 2 else None)
        if titles is not None:
            ax.set_title(titles[i], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
#Binarisation
def to_grayscale(image):
    """Accept a path, a BGR image, or an already-gray image; return uint8 grayscale."""
    if isinstance(image, str):
        image = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError('Could not read image from the given path.')
        return image
    if image.ndim == 3:
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image


def binarize(image, method='otsu', blur=True):
    """Return a binary image: white digit (255) on black background (0).

    method: 'otsu' (global, good for clean scans) or 'adaptive' (local, good for photos).
    """
    gray = to_grayscale(image)
    if blur:
        # Light Gaussian blur suppresses speckle/noise before thresholding.
        gray = cv2.GaussianBlur(gray, (3, 3), 0)

    if method == 'adaptive':
        binary = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, blockSize=31, C=10)
    else:  # 'otsu'
        _, binary = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Polarity check: MNIST wants white ink. If most pixels are white, the background is
    # white and the ink is black -> invert so the ink (digit) becomes the foreground.
    if np.mean(binary) > 127:
        binary = cv2.bitwise_not(binary)
    return binary

In [ ]:
#Deskew
def deskew(img, size=None):
    """Deskew a single grayscale digit using image moments (shear correction).

    """
    h, w = img.shape[:2]
    size = size or w
    m = cv2.moments(img)
    # mu02 ~ 0 means the digit is essentially vertical: nothing to correct.
    if abs(m['mu02']) < 1e-2:
        return img.copy()
    # skew = covariance(x,y) / variance(y)  -> the horizontal slant per unit height.
    skew = m['mu11'] / m['mu02']
    # Affine matrix that shears x by -skew*y and re-centres the result.
    M = np.float32([[1, skew, -0.5 * size * skew],
                    [0, 1, 0]])
    return cv2.warpAffine(img, M, (size, size),
                          flags=cv2.WARP_INVERSE_MAP | cv2.INTER_LINEAR)

In [ ]:
#Centre & resize
from scipy import ndimage


def _shift_to_center_of_mass(img):
    """Translate the digit so its centre of mass is at the image centre."""
    cy, cx = ndimage.center_of_mass(img)
    rows, cols = img.shape
    shiftx = np.round(cols / 2.0 - cx).astype(int)
    shifty = np.round(rows / 2.0 - cy).astype(int)
    M = np.float32([[1, 0, shiftx], [0, 1, shifty]])
    return cv2.warpAffine(img, M, (cols, rows))


def center_and_resize(binary, out_size=28, box=20):
    """Take a binary (white-on-black) image of ONE digit and return a 28x28 MNIST-style tile."""
    # 1) Crop tightly to the bounding box of the ink.
    coords = cv2.findNonZero(binary)
    if coords is None:                       # empty image -> return blank canvas
        return np.zeros((out_size, out_size), dtype=np.uint8)
    x, y, w, h = cv2.boundingRect(coords)
    digit = binary[y:y + h, x:x + w]

    # 2) Scale the longest side to `box` px, preserving aspect ratio.
    if w > h:
        new_w, new_h = box, max(1, int(round(h * box / w)))
    else:
        new_h, new_w = box, max(1, int(round(w * box / h)))
    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # 3) Paste into the centre of a black out_size x out_size canvas.
    canvas = np.zeros((out_size, out_size), dtype=np.uint8)
    x0 = (out_size - new_w) // 2
    y0 = (out_size - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = digit

    # 4) Fine-centre by centre of mass (what MNIST actually does).
    canvas = _shift_to_center_of_mass(canvas)
    return canvas

In [ ]:
#Singling digit
def preprocess_digit(image, do_deskew=True, out_size=28, flatten=False, normalize=False):
    """Full single-digit pipeline: -> binarize -> deskew -> center/resize.

    Returns a 28x28 uint8 tile, or a flat float/uint8 vector if flatten=True.
    normalize=True scales pixels to [0,1] (use the SAME choice in training & serving!).
    """
    binary = binarize(image)
    if do_deskew:
        binary = deskew(binary)
    tile = center_and_resize(binary, out_size=out_size)
    if normalize:
        tile = tile.astype(np.float32) / 255.0
    if flatten:
        return tile.reshape(-1)
    return tile

In [ ]:
#Segmentation
# def segment_digits(image, min_area=80, pad_ratio=0.25, merge_kernel=3):
#     """Locate individual digits and return (tiles, boxes) sorted left-to-right.

#     tiles : list of 28x28 normalised tiles ready for the classifier.
#     boxes : list of (x, y, w, h) bounding boxes in the ORIGINAL image coordinates,
#             handy for drawing the prediction back onto the page.
#     """
#     binary = binarize(image)

#     # Dilate slightly so a digit broken into pieces is captured as one contour.
#     if merge_kernel and merge_kernel > 1:
#         kernel = np.ones((merge_kernel, merge_kernel), np.uint8)
#         search = cv2.dilate(binary, kernel, iterations=1)
#     else:
#         search = binary

#     contours, _ = cv2.findContours(search, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

#     boxes = []
#     for c in contours:
#         x, y, w, h = cv2.boundingRect(c)
#         area = w * h
#         aspect = w / float(h)
#         # Heuristic filters: drop tiny specks and absurdly wide/flat blobs (lines, noise).
#         if area < min_area or h < 10:
#             continue
#         if aspect > 3.0:
#             continue
#         boxes.append((x, y, w, h))

#     # Read order = left to right (use y as tiebreak for multi-row pages).
#     boxes.sort(key=lambda b: (b[0]))

#     tiles = []
#     for (x, y, w, h) in boxes:
#         # Pad the crop a little so deskew/centring has breathing room.
#         pad = int(pad_ratio * max(w, h))
#         x0, y0 = max(0, x - pad), max(0, y - pad)
#         x1, y1 = min(binary.shape[1], x + w + pad), min(binary.shape[0], y + h + pad)
#         crop = binary[y0:y1, x0:x1]
#         tiles.append(preprocess_digit(crop, do_deskew=True))
#     return tiles, boxes

def segment_digits(image, min_area=80, pad_ratio=0.25, merge_kernel=1,
                   max_aspect=1.1, split_wide=True):
    """Locate individual digits and return (tiles, boxes) sorted left-to-right.

    Key fixes for "two digits captured as one":
      - merge_kernel defaults to 1 (NO dilation) so close digits stay separate.
        Dilation was bridging neighbouring strokes into a single blob.
      - Any contour wider than `max_aspect * height` is assumed to contain
        multiple touching digits and is split evenly into round(w / digit_w)
        columns (a digit is roughly as wide as it is tall).
    """
    binary = binarize(image)

    # Only dilate if explicitly asked (helps re-join a single BROKEN digit, but
    # also risks merging neighbours — off by default).
    if merge_kernel and merge_kernel > 1:
        kernel = np.ones((merge_kernel, merge_kernel), np.uint8)
        search = cv2.dilate(binary, kernel, iterations=1)
    else:
        search = binary

    contours, _ = cv2.findContours(search, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < min_area or h < 10:
            continue

        # A single digit is ~0.6*h wide. If a box is much wider than tall, it
        # most likely holds several touching digits -> split it into columns.
        if split_wide and w > max_aspect * h:
            n = max(1, int(round(w / (0.6 * h))))   # estimated digit count
            step = w / n
            for k in range(n):
                boxes.append((int(x + k * step), y, int(np.ceil(step)), h))
        else:
            boxes.append((x, y, w, h))

    # Read order = left to right (add y as a tiebreak for multi-row pages).
    boxes.sort(key=lambda b: (b[0]))

    tiles = []
    for (x, y, w, h) in boxes:
        pad = int(pad_ratio * max(w, h))
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(binary.shape[1], x + w + pad), min(binary.shape[0], y + h + pad)
        crop = binary[y0:y1, x0:x1]
        tiles.append(preprocess_digit(crop, do_deskew=True))
    return tiles, boxes

In [ ]:
#Make demo
def make_demo_image(text='51403', size=(160, 700)):
    """Render digits with varying slant/scale/jitter to mimic a handwritten strip."""
    img = np.zeros(size, dtype=np.uint8)
    x = 40
    rng = np.random.default_rng(0)
    for ch in text:
        scale = rng.uniform(3.5, 5.0)
        thick = rng.integers(6, 10)
        y = rng.integers(95, 120)
        cv2.putText(img, ch, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, 255, int(thick),
                    lineType=cv2.LINE_AA)
        x += int(70 * scale / 4)
    # Add a touch of speckle noise like a real scan.
    noise = (rng.random(size) < 0.01).astype(np.uint8) * 255
    img = cv2.bitwise_or(img, noise)
    return img


demo = make_demo_image('51403')
show([demo], titles=['Synthetic handwritten strip (input)'], cols=1, figsize=(10, 3))

In [ ]:
#Demo segmentaion
# Run segmentation, then visualise the detected boxes and the normalised tiles.
tiles, boxes = segment_digits(demo)
print(f'Detected {len(tiles)} digit(s).')

vis = cv2.cvtColor(demo, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in boxes:
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)
show([vis], titles=['Detected digit bounding boxes'], cols=1, figsize=(10, 3))

if tiles:
    show(tiles, titles=[f'tile {i}' for i in range(len(tiles))], cols=len(tiles),
         figsize=(2 * len(tiles), 2.5))

In [ ]:
#Demo single digil
raw = make_demo_image('7', size=(120, 120))
b = binarize(raw)
d = deskew(b)
final = center_and_resize(d)
show([raw, b, d, final],
     titles=['1. raw', '2. binarized', '3. deskewed', '4. centered 28x28'],
     cols=4, figsize=(11, 3))

In [ ]:
#Export the pipeline
module_code = r'''"""
digit_preprocessing.py  (auto-generated by Notebook 1)

Shared OpenCV preprocessing for handwritten-digit recognition.
Used IDENTICALLY at training time (Notebook 2) and at inference/deployment time
to eliminate train/serve skew.
"""
import cv2
import numpy as np
from scipy import ndimage


def to_grayscale(image):
    if isinstance(image, str):
        image = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError('Could not read image from the given path.')
        return image
    if image.ndim == 3:
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image


def binarize(image, method='otsu', blur=True):
    gray = to_grayscale(image)
    if blur:
        gray = cv2.GaussianBlur(gray, (3, 3), 0)
    if method == 'adaptive':
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 31, 10)
    else:
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(binary) > 127:
        binary = cv2.bitwise_not(binary)
    return binary


def deskew(img, size=None):
    h, w = img.shape[:2]
    size = size or w
    m = cv2.moments(img)
    if abs(m['mu02']) < 1e-2:
        return img.copy()
    skew = m['mu11'] / m['mu02']
    M = np.float32([[1, skew, -0.5 * size * skew], [0, 1, 0]])
    return cv2.warpAffine(img, M, (size, size),
                          flags=cv2.WARP_INVERSE_MAP | cv2.INTER_LINEAR)


def _shift_to_center_of_mass(img):
    cy, cx = ndimage.center_of_mass(img)
    rows, cols = img.shape
    shiftx = np.round(cols / 2.0 - cx).astype(int)
    shifty = np.round(rows / 2.0 - cy).astype(int)
    M = np.float32([[1, 0, shiftx], [0, 1, shifty]])
    return cv2.warpAffine(img, M, (cols, rows))


def center_and_resize(binary, out_size=28, box=20):
    coords = cv2.findNonZero(binary)
    if coords is None:
        return np.zeros((out_size, out_size), dtype=np.uint8)
    x, y, w, h = cv2.boundingRect(coords)
    digit = binary[y:y + h, x:x + w]
    if w > h:
        new_w, new_h = box, max(1, int(round(h * box / w)))
    else:
        new_h, new_w = box, max(1, int(round(w * box / h)))
    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((out_size, out_size), dtype=np.uint8)
    x0 = (out_size - new_w) // 2
    y0 = (out_size - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = digit
    return _shift_to_center_of_mass(canvas)


def preprocess_digit(image, do_deskew=True, out_size=28, flatten=False, normalize=False):
    binary = binarize(image)
    if do_deskew:
        binary = deskew(binary)
    tile = center_and_resize(binary, out_size=out_size)
    if normalize:
        tile = tile.astype(np.float32) / 255.0
    if flatten:
        return tile.reshape(-1)
    return tile


def segment_digits(image, min_area=80, pad_ratio=0.25, merge_kernel=1,
                   max_aspect=1.1, split_wide=True):
    """Locate individual digits and return (tiles, boxes) sorted left-to-right.

    Key fixes for "two digits captured as one":
      - merge_kernel defaults to 1 (NO dilation) so close digits stay separate.
        Dilation was bridging neighbouring strokes into a single blob.
      - Any contour wider than `max_aspect * height` is assumed to contain
        multiple touching digits and is split evenly into round(w / digit_w)
        columns (a digit is roughly as wide as it is tall).
    """
    binary = binarize(image)

    # Only dilate if explicitly asked (helps re-join a single BROKEN digit, but
    # also risks merging neighbours — off by default).
    if merge_kernel and merge_kernel > 1:
        kernel = np.ones((merge_kernel, merge_kernel), np.uint8)
        search = cv2.dilate(binary, kernel, iterations=1)
    else:
        search = binary

    contours, _ = cv2.findContours(search, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < min_area or h < 10:
            continue

        # A single digit is ~0.6*h wide. If a box is much wider than tall, it
        # most likely holds several touching digits -> split it into columns.
        if split_wide and w > max_aspect * h:
            n = max(1, int(round(w / (0.6 * h))))   # estimated digit count
            step = w / n
            for k in range(n):
                boxes.append((int(x + k * step), y, int(np.ceil(step)), h))
        else:
            boxes.append((x, y, w, h))

    # Read order = left to right (add y as a tiebreak for multi-row pages).
    boxes.sort(key=lambda b: (b[0]))

    tiles = []
    for (x, y, w, h) in boxes:
        pad = int(pad_ratio * max(w, h))
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(binary.shape[1], x + w + pad), min(binary.shape[0], y + h + pad)
        crop = binary[y0:y1, x0:x1]
        tiles.append(preprocess_digit(crop, do_deskew=True))
    return tiles, boxes
'''

with open('digit_preprocessing.py', 'w') as f:
    f.write(module_code)
print('Wrote digit_preprocessing.py (', len(module_code), 'bytes ).')

# Smoke-test the freshly written module so we know it imports cleanly.
import importlib, digit_preprocessing as dp
importlib.reload(dp)
_t = dp.preprocess_digit(make_demo_image('9', size=(120, 120)))
print('Module import OK. Sample tile shape:', _t.shape, 'dtype:', _t.dtype)